In [1]:
import numpy
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Conv2D, Input, Activation, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Model
from keras.utils import to_categorical

In [ ]:

(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [3]:
IMG_SHAPE = (28,28,1)

X_train = X_train / 255
X_test = X_test / 255

In [4]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
num_classes = y_test.shape[1]

In [5]:
def baseline_model(input_shape):
    # create model
    X_input = Input(input_shape)
    X = Conv2D(32,(5,5),strides=(1,1),name='conv0')(X_input)
    X = Activation('relu')(X)
    X = MaxPooling2D((2, 2), name='max_pool0')(X)
    X = Flatten()(X)
    X = Dense(128,activation='relu',name='fc1')(X)
    X = Dense(num_classes, activation='softmax')(X)
    model = Model(inputs = X_input, outputs = X, name='Simple_MnistNet')
    # Compile model
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [ ]:

model = baseline_model(IMG_SHAPE)
model.summary()

In [ ]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=200, verbose=1)

In [ ]:
scores = model.evaluate(X_test, y_test, verbose=0)
print("CNN Error: %.2f%%" % (100-scores[1]*100))

In [ ]:
!git clone https://github.com/deepanrajm/GL_MML.git

In [ ]:
img = tf.keras.utils.load_img("/content/GL_MML/Unit_2/Visual_Representation/2.png", target_size=(28,28),color_mode='grayscale')
img_array = tf.keras.utils.img_to_array(img)
print(img_array.shape)
img_array =  numpy.expand_dims(img_array, axis = 0)
img_array = img_array / 255

predictions = model.predict(img_array)
print (numpy.argmax(predictions))

In [ ]:
import matplotlib.pyplot as plt

layer_outputs = [layer.output for layer in model.layers if 'conv' in layer.name]
activation_model = Model(inputs=model.input, outputs=layer_outputs)

activations = activation_model.predict(img_array)

plt.figure(figsize=(15, 15))
for i in range(32):
    plt.subplot(6, 6, i+1)
    plt.imshow(activations[0, :, :, i], cmap='viridis')
    plt.axis('off')
    plt.title(f"Filter {i}")
plt.show()

In [ ]:
img = tf.keras.utils.load_img("/content/GL_MML/Unit_2/Visual_Representation/elephant.jpg", target_size=(28,28),color_mode='grayscale')
img_array = tf.keras.utils.img_to_array(img)
print(img_array.shape)
img_array =  numpy.expand_dims(img_array, axis = 0)
img_array = img_array / 255

predictions = model.predict(img_array)
print (numpy.argmax(predictions))

import matplotlib.pyplot as plt

layer_outputs = [layer.output for layer in model.layers if 'conv' in layer.name]
activation_model = Model(inputs=model.input, outputs=layer_outputs)

activations = activation_model.predict(img_array)

plt.figure(figsize=(15, 15))
for i in range(32):
    plt.subplot(6, 6, i+1)
    plt.imshow(activations[0, :, :, i], cmap='viridis')
    plt.axis('off')
    plt.title(f"Filter {i}")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model

model = VGG16(weights='imagenet')

img_path = "/content/GL_MML/Unit_2/Visual_Representation/elephant.jpg"
img = image.load_img(img_path, target_size=(224, 224))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

preds = model.predict(x)
print("--------------------------------------------------")
print("AI CLASSIFICATION RESULT:")

for i, (id, label, prob) in enumerate(decode_predictions(preds, top=3)[0]):
    print(f"{i+1}. {label}: {prob*100:.2f}%")
print("--------------------------------------------------")

layer_names = ['block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1', 'block5_conv1']
intermediate_outputs = [model.get_layer(name).output for name in layer_names]
vis_model = Model(inputs=model.input, outputs=intermediate_outputs)

feature_maps = vis_model.predict(x)

fig, axes = plt.subplots(1, 7, figsize=(25, 5))

axes[0].imshow(img)
axes[0].set_title("Original Input")
axes[0].axis('off')

for i, fmap in enumerate(feature_maps):

    axes[i+1].imshow(fmap[0, :, :, 10], cmap='viridis')
    axes[i+1].set_title(f"Representation\n(Block {i+1})")
    axes[i+1].axis('off')

plt.show()